In [2]:
#Task 1 – Load 15–20 business facts into ChromaDB
business_facts = [
    "Business name: Replace with your business name.",
    "Business location: Replace with your business location.",
    "Business type: Replace with your business category.",
    "Products: List the products offered by the business.",
    "Services: List the services offered by the business.",
    "Business history: Explain when the business started.",
    "Business owner: Mention the business owner.",
    "Working hours: Mention the opening and closing times.",
    "Contact details: Add the official business contact details.",
    "Pricing: Explain the product or service pricing.",
    "Payment methods: Mention accepted payment methods.",
    "Customer support: Explain how customers receive support.",
    "Delivery: Explain the delivery or collection process.",
    "Special offers: Mention available discounts or offers.",
    "Business policies: Explain the important business policies."
]


In [4]:
!pip install -q langchain-chroma langchain-huggingface langchain-xai langgraph sentence-transformers

In [5]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_xai import ChatXAI
from langgraph.graph import StateGraph

print("All project imports successful!")

All project imports successful!


In [7]:
business_facts = [
    "Business name: ABC Store",
    "Business location: Hyderabad",
    "Business type: Retail business",
    "Products: Electronics and accessories",
    "Services: Product sales and customer support",
    "Business history: Started in 2020",
    "Business owner: ABC",
    "Working hours: 9 AM to 9 PM",
    "Contact details: 9876543210",
    "Pricing: Depends on the product",
    "Payment methods: Cash and UPI",
    "Customer support: Available during working hours",
    "Delivery: Available within the city",
    "Special offers: Seasonal discounts",
    "Business policies: Products can be exchanged within 7 days"
]

print("Total facts:", len(business_facts))

Total facts: 15


In [10]:
%%writefile vector_store.py

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

from business_facts import business_facts

# Initialize embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create ChromaDB
vector_store = Chroma(
    collection_name="business_knowledge",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
    collection_metadata={"hnsw:space": "cosine"}
)

# Convert facts to documents
documents = []

for index, fact in enumerate(business_facts):

    document = Document(
        page_content=fact,
        metadata={
            "fact_number": index + 1,
            "source": "business_homework"
        }
    )

    documents.append(document)

# Unique IDs
ids = [
    f"business_fact_{i+1}"
    for i in range(len(documents))
]

# Store documents
vector_store.add_documents(
    documents=documents,
    ids=ids
)

print("Business facts successfully stored!")
print("Total facts:", len(documents))
print("Total chunks:", vector_store._collection.count())

Writing vector_store.py


In [13]:
#Task 3 – Add the search tool to your Day 4 agent with a checkpointer


%%writefile search_tool.py

from langchain.tools import tool
from vector_store import vector_store


@tool
def search_business_knowledge(query: str) -> str:
    """
    Search the business knowledge database.

    Topics covered:
    1. Business name and location
    2. Business type and history
    3. Products and services
    4. Business owner
    5. Working hours
    6. Contact details
    7. Pricing and payment methods
    8. Customer support
    9. Delivery and collection
    10. Special offers and discounts
    11. Business policies

    Use this tool for questions related to
    the stored business information.
    """

    results = vector_store.similarity_search_with_score(
        query,
        k=3
    )

    relevant_results = [
        doc.page_content
        for doc, score in results
        if score <= 0.45
    ]

    if not relevant_results:
        return (
            "NO_RELEVANT_INFORMATION: "
            "The business knowledge database does not "
            "contain sufficient information."
        )

    return "\n".join(relevant_results)


print("Business search tool created successfully!")


Writing search_tool.py


In [14]:
#Task 4 – Break the agent and fix hallucinations
RULES = """
You are a business information assistant.

Your answers must be based on the business
knowledge database only.

STRICT RULES:

1. For every business-related question,
   call search_business_knowledge first.

2. Never invent facts, prices, names,
   phone numbers, revenue or other details.

3. Never use your general knowledge to fill
   missing business information.

4. If the search tool returns
   NO_RELEVANT_INFORMATION, do not attempt
   to guess the answer.

5. If information is missing, respond:

   "Sorry, I don't have that information
   in my business knowledge database.
   Please check with the business directly."

6. If only part of the answer is available,
   answer only that part and clearly mention
   which details are unavailable.

7. Do not treat previous guesses as facts.

8. Keep responses short and accurate.
"""


In [18]:
%%writefile business_facts.py

business_facts = [
    "Business name: ABC Store",
    "Business location: Hyderabad",
    "Business type: Retail business",
    "Products: Electronics and accessories",
    "Services: Product sales and customer support",
    "Business history: Started in 2020",
    "Business owner: ABC",
    "Working hours: 9 AM to 9 PM",
    "Contact details: 9876543210",
    "Pricing: Depends on the product",
    "Payment methods: Cash and UPI",
    "Customer support: Available during working hours",
    "Delivery: Available within the city",
    "Special offers: Seasonal discounts",
    "Business policies: Products can be exchanged within 7 days"
]

print("Business facts loaded:", len(business_facts))

Writing business_facts.py


In [20]:
import os

print(os.getcwd())
print(os.listdir("/content"))

/content
['.config', 'chroma_db', 'search_tool.py', '__pycache__', 'business_facts.py', 'vector_store.py', 'sample_data']


In [23]:
import os
from getpass import getpass

os.environ["XAI_API_KEY"] = getpass("Enter your Grok API key: ")

Enter your Grok API key: ··········


In [24]:
import os

if os.getenv("XAI_API_KEY"):
    print("Grok API key configured successfully!")
else:
    print("API key is missing!")

Grok API key configured successfully!


In [25]:
from langchain_xai import ChatXAI

llm = ChatXAI(
    model="grok-3-mini",
    temperature=0
)

print("Grok model initialized successfully!")

Grok model initialized successfully!


In [26]:
from langchain_xai import ChatXAI
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

from search_tool import search_business_knowledge

# Initialize Grok
llm = ChatXAI(
    model="grok-3-mini",
    temperature=0
)

# Register the tool
tools = [search_business_knowledge]
llm_with_tools = llm.bind_tools(tools)

# Agent rules
RULES = """
You are a business knowledge assistant.

1. Use the search_business_knowledge tool for business-related questions.
2. Answer using only information from the business knowledge database.
3. Never invent or guess business information.
4. If the database does not contain the answer, clearly say:
   I don't have sufficient information about this in my database.
"""

# Define agent node
def call_model(state):
    messages = [SystemMessage(content=RULES)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# Build graph
builder = StateGraph(MessagesState)

builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

# Compile with checkpointer
checkpointer = InMemorySaver()
agent = builder.compile(checkpointer=checkpointer)

print("Agent created successfully!")

Agent created successfully!


In [32]:
from google.colab import userdata
import os

api_key = userdata.get("GROQ_API_KEY")
os.environ["XAI_API_KEY"] = api_key

print("API key loaded:", bool(api_key))

API key loaded: True


In [36]:
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")

print("Key loaded:", bool(api_key))
print("Key prefix:", api_key[:4] if api_key else "Missing")
print("Key length:", len(api_key) if api_key else 0)

Key loaded: True
Key prefix: gsk_
Key length: 56


In [37]:
!pip install -q langchain-groq

In [40]:
from google.colab import userdata

groq_api_key = userdata.get("GROQ_API_KEY")

print("API key loaded:", bool(groq_api_key))

API key loaded: True


In [43]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key=groq_api_key,
    model="openai/gpt-oss-120b",
    temperature=0
)

print("Groq model initialized successfully!")

Groq model initialized successfully!


In [44]:
response = llm.invoke(
    "Explain what a vector database is in two sentences."
)

print(response.content)

A vector database is a specialized data store that indexes and retrieves high‑dimensional numeric vectors—often embeddings generated from text, images, audio, or other data—using similarity search algorithms (e.g., approximate nearest neighbor). It enables fast, scalable queries like “find the most similar items to this vector,” which is essential for AI‑driven applications such as semantic search, recommendation, and anomaly detection.


In [48]:
from google.colab import userdata
from langchain_groq import ChatGroq

groq_api_key = userdata.get("GROQ_API_KEY")

groq_llm = ChatGroq(
    api_key=groq_api_key,
    model="openai/gpt-oss-120b",
    temperature=0
)

print(type(groq_llm))

<class 'langchain_groq.chat_models.ChatGroq'>


In [49]:
response = groq_llm.invoke(
    "Say hello in one sentence."
)

print(response.content)

Hello! I hope you're having a wonderful day.


In [50]:
llm_with_tools = groq_llm.bind_tools(tools)

In [51]:
#Task 5 – Test with two different thread IDs
thread_id = "user_1"

config = {
    "configurable": {
        "thread_id": thread_id
    }
}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What products does the business offer?"}]},
    config=config
)

print(response["messages"][-1].content)

I’m happy to help, but I need to know which specific business you’re asking about. Could you please tell me the name of the business (or any other identifying details) so I can look up its product offerings?


In [55]:

%%writefile agent.py

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

from google.colab import userdata
from search_tool import search_business_knowledge

# Initialize Groq
groq_api_key = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    api_key=groq_api_key,
    model="openai/gpt-oss-120b",
    temperature=0
)

tools = [search_business_knowledge]
llm_with_tools = llm.bind_tools(tools)

RULES = """
You are a business knowledge assistant.

1. Use the search tool for business-related questions.
2. Answer only using information from the business knowledge database.
3. Never invent business information.
4. If information is unavailable, clearly say:
   I don't have sufficient information in my database.
"""

def call_model(state):
    messages = [SystemMessage(content=RULES)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

builder = StateGraph(MessagesState)

builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

checkpointer = InMemorySaver()
agent = builder.compile(checkpointer=checkpointer)

def chat(question, thread_id):
    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    response = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config
    )

    return response["messages"][-1].content

print("Agent file created successfully!")


Overwriting agent.py


In [56]:
from agent import chat

print("Agent imported successfully!")

Agent file created successfully!
Agent imported successfully!


In [57]:
answer = chat(
    "What products does the business offer?",
    "user_1"
)

print(answer)

I don't have sufficient information in my database.


In [59]:
#Verify chromaDB storage
from vector_store import vector_store

total_chunks = vector_store._collection.count()

print("Total business facts:", total_chunks)

if total_chunks >= 15:
    print("PASS: Required business facts are stored.")
else:
    print("FAIL: At least 15 facts are required.")

Total business facts: 15
PASS: Required business facts are stored.


In [63]:
#Verify the search tool
from search_tool import search_business_knowledge

print(search_business_knowledge.description)

Search the business knowledge database.

Topics covered:
1. Business name and location
2. Business type and history
3. Products and services
4. Business owner
5. Working hours
6. Contact details
7. Pricing and payment methods
8. Customer support
9. Delivery and collection
10. Special offers and discounts
11. Business policies

Use this tool for questions related to
the stored business information.


In [61]:
result = search_business_knowledge.invoke({
    "query": "What products does the business offer?"
})

print(result)

Business type: Retail business


In [62]:
questions = [
    "Where is the business located?",
    "What payment methods are accepted?",
    "What are the business working hours?"
]

for question in questions:
    print("\nQuestion:", question)
    print("Answer:", search_business_knowledge.invoke({"query": question}))


Question: Where is the business located?
Answer: Business location: Hyderabad
Business name: ABC Store

Question: What payment methods are accepted?
Answer: Payment methods: Cash and UPI

Question: What are the business working hours?
Answer: Working hours: 9 AM to 9 PM


In [64]:
#Test agent integration
from agent import chat

answer = chat(
    "What products does the business offer?",
    "business_test_1"
)

print(answer)

I don't have sufficient information in my database.


In [65]:
#Test conversation memory
answer1 = chat(
    "My favourite colour is blue. Remember this.",
    "memory_test_1"
)

print(answer1)

I’m focused on providing information from the business knowledge database, so I don’t retain personal preferences like favorite colours. If you have any business‑related questions, feel free to ask!


In [66]:
answer2 = chat(
    "What is my favourite colour?",
    "memory_test_1"
)

print(answer2)

I don’t have that information stored in my database. If you have any business‑related questions, I’d be happy to help!


In [67]:
#Test thread isolation
answer3 = chat(
    "What is my favourite colour?",
    "memory_test_2"
)

print(answer3)

I don't have sufficient information in my database.


In [68]:
#Test hallucination prevention
unknown_questions = [
    "What is the owner's favourite food?",
    "What was the business revenue last year?",
    "How many employees work in the business?"
]

for question in unknown_questions:
    print("\nQuestion:", question)
    answer = chat(question, "unknown_test_1")
    print("Answer:", answer)


Question: What is the owner's favourite food?
Answer: I don't have sufficient information in my database.

Question: What was the business revenue last year?
Answer: I don't have sufficient information in my database.I don't have sufficient information in my database.

Question: How many employees work in the business?
Answer: I ​​​​​​​ ​  ​  ​  ​  ​  ​ ​  ​ ​ ​ ​  ​  ​ ​ ​​ ​ ​ ​ ​ ​ ​​ ​ ​​ ​ ​​ ​ ​​​  ​ ​ ​ ​​​ ​ ​​​ ​ ​​​​ ​ ​​ ​​​​​ ​ ​​​​ ​​​​​​​ ​ ​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​​ 

I’m sorry…  



It looks  







... 























...



...











...



...



...



We 







...



...



...



...



...



...



...



...



...



...



We











...



The 



...



...



...



...



...



...



...



...



...



...



...


In [69]:
#Improve your agent RULES
RULES = """
You are a business knowledge assistant.

STRICT RULES:

1. Use the search_business_knowledge tool
   for every business-related question.

2. Answer business questions only using
   information retrieved from the database.

3. Never invent, assume, or guess business facts.

4. If the tool returns
   NO_RELEVANT_INFORMATION, do not provide
   an answer based on your own knowledge.

5. Clearly say:
   "I don't have sufficient information
   about this in my business database."

6. You may use previous conversation messages
   to answer questions about information
   explicitly shared within the same thread.

7. Never use information from another thread.

8. If the user asks an unrelated general
   knowledge question, explain that you are
   designed to answer questions about the
   stored business information.
"""